# Prévision Multi-Actifs par Deep Learning
## Actions · Crypto · Forex · Matières Premières · Indices

---
**Objectif :** Entraîner des modèles LSTM, BiLSTM et Transformer sur chaque grande classe d'actifs et comparer leurs performances.

| Classe d'actif | Exemples | Caractéristiques |
|---|---|---|
| Actions US | TSLA, NVDA, MSFT | Forte volatilité, sensibles aux résultats |
| Crypto | BTC, ETH, SOL | 24/7, volatilité extrême |
| Forex | EUR/USD, GBP/JPY | Faible volatilité, macro-sensible |
| Matières premières | Or, Pétrole, Argent | Couverture inflation, offre/demande |
| Indices | S&P500, NASDAQ, DAX | Marché global, diversifié |

**Question clé :** *Quelle classe d'actif est la plus prévisible par le deep learning ?*

## 1. Installation & Imports

In [12]:
import subprocess, sys
for pkg in ['yfinance','ta','torch','scikit-learn','pandas','numpy',
            'matplotlib','seaborn','plotly','tqdm','scipy']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
print(' Packages prêts !')

 Packages prêts !


In [13]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import yfinance as yf
import ta
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from tqdm import tqdm
import math, os, json, copy
import scipy.stats as stats

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositif : {device} | PyTorch : {torch.__version__} | Pandas : {pd.__version__}')

Dispositif : cuda | PyTorch : 2.10.0+cu128 | Pandas : 2.2.2


## 2. Univers d'Actifs & Configuration

In [14]:
ASSET_UNIVERSE = {
    'Actions':        {'TSLA': 'Tesla',       'NVDA': 'NVIDIA',        'MSFT': 'Microsoft',
                       'JPM':  'JPMorgan',    'XOM':  'ExxonMobil'},
    'Crypto':         {'BTC-USD': 'Bitcoin',  'ETH-USD': 'Ethereum',   'SOL-USD': 'Solana',
                       'BNB-USD': 'Binance Coin'},
    'Forex':          {'EURUSD=X': 'EUR/USD', 'GBPUSD=X': 'GBP/USD',  'USDJPY=X': 'USD/JPY',
                       'AUDUSD=X': 'AUD/USD'},
    'Matières Prem.': {'GC=F': 'Or',          'CL=F':   'Pétrole brut','SI=F': 'Argent',
                       'NG=F': 'Gaz naturel'},
    'Indices':        {'^GSPC': 'S&P 500',    '^IXIC':  'NASDAQ',     '^GDAXI': 'DAX',
                       '^N225': 'Nikkei 225'},
}
ALL_TICKERS = {t: (cls, name)
               for cls, assets in ASSET_UNIVERSE.items()
               for t, name in assets.items()}

CONFIG = {
    'date_debut':    '2019-01-01',
    'date_fin':      '2026-05-15',
    'intervalle':    '1d',
    'seq_len':       60,
    'horizon':       1,          # prédire le rendement J+1
    'cible':         'Ret_Cible',
    'epochs':        50,
    'batch_size':    32,
    'lr':            0.001,
    'hidden_size':   128,
    'num_layers':    2,
    'dossier':       'resultats_multi_actifs/',
}
os.makedirs(CONFIG['dossier'], exist_ok=True)

COULEURS_CLASSES = {
    'Actions': '#3498db', 'Crypto': '#f39c12',
    'Forex': '#2ecc71',   'Matières Prem.': '#e74c3c', 'Indices': '#9b59b6'
}
print(' Configuration chargée.')

 Configuration chargée.


## 3. Téléchargement & Validation des Données

In [15]:
def telecharger_actif(ticker, cfg):
    """Télécharge et valide un actif. Retourne (DataFrame, None) ou (None, erreur)."""
    try:
        df = yf.download(ticker, start=cfg['date_debut'], end=cfg['date_fin'],
                         interval=cfg['intervalle'], progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df.dropna(inplace=True)
        df.index = pd.to_datetime(df.index)
        if len(df) < 300:
            return None, f'Seulement {len(df)} lignes'
        return df, None
    except Exception as e:
        return None, str(e)

donnees_brutes = {}
journal = []

print('Téléchargement de tous les actifs...\n')
for ticker, (classe, nom) in ALL_TICKERS.items():
    df, err = telecharger_actif(ticker, CONFIG)
    statut = '' if df is not None else ''
    lignes = len(df) if df is not None else 0
    print(f'  {statut} {ticker:<12} {nom:<25} {lignes:>4} lignes  {err or ""}')
    if df is not None:
        donnees_brutes[ticker] = df
    journal.append({'ticker': ticker, 'classe': classe, 'nom': nom,
                    'lignes': lignes, 'ok': df is not None})

n_ok = sum(d['ok'] for d in journal)
print(f'\n {n_ok}/{len(journal)} actifs téléchargés avec succès')

Téléchargement de tous les actifs...

   TSLA         Tesla                     1852 lignes  
   NVDA         NVIDIA                    1852 lignes  
   MSFT         Microsoft                 1852 lignes  
   JPM          JPMorgan                  1852 lignes  
   XOM          ExxonMobil                1852 lignes  
   BTC-USD      Bitcoin                   2691 lignes  
   ETH-USD      Ethereum                  2691 lignes  
   SOL-USD      Solana                    2226 lignes  
   BNB-USD      Binance Coin              2691 lignes  
   EURUSD=X     EUR/USD                   1917 lignes  
   GBPUSD=X     GBP/USD                   1917 lignes  
   USDJPY=X     USD/JPY                   1917 lignes  
   AUDUSD=X     AUD/USD                   1917 lignes  
   GC=F         Or                        1854 lignes  
   CL=F         Pétrole brut              1854 lignes  
   SI=F         Argent                    1854 lignes  
   NG=F         Gaz naturel               1855 lignes  
   ^GSPC  

## 3.1 Aperçu Visuel — Tous les Actifs (Base 100)

In [16]:
num_lignes = math.ceil(len(donnees_brutes) / 4)
fig = make_subplots(
    rows=num_lignes, cols=4,
    subplot_titles=[f"{t}\n{ALL_TICKERS[t][1]}" for t in donnees_brutes],
    vertical_spacing=0.08, horizontal_spacing=0.05
)

for idx, (ticker, df) in enumerate(donnees_brutes.items()):
    r, c = divmod(idx, 4)
    couleur = COULEURS_CLASSES[ALL_TICKERS[ticker][0]]
    norme = (df['Close'] / df['Close'].iloc[0]) * 100
    fig.add_trace(go.Scatter(x=df.index, y=norme, name=ticker,
                             line=dict(color=couleur, width=1.2), showlegend=False),
                  row=r+1, col=c+1)

for cls, couleur in COULEURS_CLASSES.items():
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                             name=cls, line=dict(color=couleur, width=3)))

fig.update_layout(
    title='Tous les actifs — Prix normalisé (Base=100) depuis 2019',
    height=num_lignes * 320, template='plotly_dark',
    legend=dict(orientation='h', y=-0.05)
)
fig.update_xaxes(showticklabels=False)
fig.show()

## 4. Ingénierie de Caractéristiques
> Forex : le volume est ignoré (peu fiable). Toutes les cibles sont des **rendements logarithmiques** (série stationnaire).

In [17]:
def ajouter_features(df, classe):
    """Calcule les indicateurs techniques par classe d'actif."""
    df = df.copy()
    df['Close'] = df['Close'].replace(0, np.nan).ffill().bfill()

    # Cible : rendement logarithmique J+1
    df['Ret_Cible'] = np.log(df['Close'] / df['Close'].shift(1))

    # Volatilité & momentum
    vol_20 = df['Ret_Cible'].rolling(20).std().replace(0, np.nan).ffill() + 1e-9
    df['Vol_Norm_Ret'] = df['Ret_Cible'] / vol_20
    df['RSI'] = ta.momentum.rsi(df['Close'], window=14).fillna(50) / 100.0

    atr = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'])
    df['ATR_Ratio'] = (atr / df['Close']).fillna(0)

    for w in [20, 50]:
        sma = df['Close'].rolling(w).mean()
        std = df['Close'].rolling(w).std().replace(0, np.nan).ffill() + 1e-9
        df[f'Dist_SMA_{w}'] = (df['Close'] - sma) / std

    df['Mom_Ret'] = df['Ret_Cible'].rolling(5).mean()

    # Régime de volatilité
    real_vol = df['Ret_Cible'].rolling(20).std().fillna(0)
    df['Regime_Vol'] = pd.qcut(real_vol, 3, labels=[0,1,2],
                                duplicates='drop').fillna(1).astype(int)

    # Volume (hors Forex)
    if classe != 'Forex' and 'Volume' in df.columns:
        df['Delta_Vol'] = df['Volume'].pct_change().replace([np.inf,-np.inf], 0).fillna(0)

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    cols_num = df.select_dtypes(include=[np.number]).columns
    df[cols_num] = df[cols_num].clip(-1e5, 1e5)
    return df

print(' Fonction de features définie.')

 Fonction de features définie.


## 5. Préparation des Données (sans fuite temporelle)

In [18]:
class JeuDeDonnees(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


def preparer_donnees(df, cfg, idx_train, idx_val, idx_test):
    """Crée les séquences sans fuite : le scaler est ajusté uniquement sur train."""
    cols_feats = [c for c in df.columns
                  if c not in ['Close','Dividends','Stock Splits','Adj Close']]
    seq_len = cfg['seq_len']
    horizon = cfg['horizon']

    df_tr = df.iloc[idx_train]
    df_vl = df.iloc[idx_val]
    df_te = df.iloc[idx_test]

    scaler = RobustScaler().fit(df_tr[cols_feats])

    def creer_sequences(d):
        X_sc = scaler.transform(d[cols_feats])
        y_sc = d[[cfg['cible']]].values
        Xs, ys, cibles = [], [], []
        for i in range(seq_len, len(d) - horizon):
            Xs.append(X_sc[i-seq_len:i])
            ys.append(y_sc[i:i+horizon].flatten())
            cibles.append(d[cfg['cible']].iloc[i-1])
        return np.array(Xs), np.array(ys), np.array(cibles)

    X_tr,y_tr,_    = creer_sequences(df_tr)
    X_vl,y_vl,_    = creer_sequences(df_vl)
    X_te,y_te,c_te = creer_sequences(df_te)

    return {
        'train':      JeuDeDonnees(X_tr, y_tr),
        'val':        JeuDeDonnees(X_vl, y_vl),
        'test':       JeuDeDonnees(X_te, y_te),
        'cibles_test': c_te,       # vrais rendements log du test
        'n_feats':    len(cols_feats),
        'scaler':     scaler,
        'cols_feats': cols_feats,
    }

print(' Fonctions de préparation définies.')

 Fonctions de préparation définies.


## 6. Architectures — LSTM · BiLSTM · Transformer

In [19]:
class ModeleLSTM(nn.Module):
    def __init__(self, input_size, hidden=128, layers=2, output=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, layers, batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hidden, output)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class ModeleBiLSTM(nn.Module):
    def __init__(self, input_size, hidden=128, layers=2, output=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, layers, batch_first=True,
                            dropout=dropout, bidirectional=True)
        self.fc   = nn.Linear(hidden * 2, output)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class ModeleTransformer(nn.Module):
    def __init__(self, input_size, d_model=128, nhead=4, layers=2, output=1, dropout=0.2):
        super().__init__()
        self.proj     = nn.Linear(input_size, d_model)
        enc_layer     = nn.TransformerEncoderLayer(d_model, nhead, 256, dropout, batch_first=True)
        self.encoder  = nn.TransformerEncoder(enc_layer, layers)
        self.fc       = nn.Linear(d_model, output)
    def forward(self, x):
        x = self.proj(x)
        x = self.encoder(x)
        return self.fc(x[:, -1, :])

print(' Architectures définies (LSTM, BiLSTM, Transformer).')

 Architectures définies (LSTM, BiLSTM, Transformer).


## 7. Fonctions Utilitaires

In [20]:
#  Test de Pesaran-Timmermann
def test_pt(y_vrai, y_pred):
    """Teste la significativité statistique de la précision directionnelle."""
    d_v = (y_vrai > 0).astype(float)
    d_p = (y_pred > 0).astype(float)
    n   = len(d_v)
    p_h   = np.mean(d_v == d_p)
    p_y   = np.mean(d_v)
    p_x   = np.mean(d_p)
    p_star = p_y*p_x + (1-p_y)*(1-p_x)
    v_h   = p_star*(1-p_star)/n
    v_star = (((2*p_y-1)**2 * p_x*(1-p_x)) +
              ((2*p_x-1)**2 * p_y*(1-p_y)) +
              4*p_y*p_x*(1-p_y)*(1-p_x)/n) / n
    z     = (p_h - p_star) / np.sqrt(max(v_h - v_star, 1e-12))
    p_val = 1 - stats.norm.cdf(abs(z))
    return p_h, p_val


#  Backtest amélioré
def backtest(pred_ret, prix_reels, capital=10_000, cout=0.001):
    """
    Backtest sans biais de regard en avant.
    - pred_ret  : rendements log prédits (signal J)
    - prix_reels: prix de clôture réels alignés sur le test
    Signal positif → achat exécuté au prix du jour suivant.
    """
    cash, pos = capital, 0.0
    portefeuille = [capital]
    bah          = [capital]   # buy & hold

    for i in range(len(pred_ret) - 1):
        prix_exec = prix_reels[i+1]  # exécution J+1
        if pred_ret[i] > 0 and cash > 0:
            pos  = (cash * (1 - cout)) / prix_exec
            cash = 0
        elif pred_ret[i] <= 0 and pos > 0:
            cash = pos * prix_exec * (1 - cout)
            pos  = 0
        portefeuille.append(cash + pos * prix_exec)

    # Buy & Hold
    bah_pos = capital / prix_reels[0]
    bah     = [capital] + [bah_pos * p for p in prix_reels[1:]]

    pf = np.array(portefeuille[:len(prix_reels)])
    bh = np.array(bah[:len(prix_reels)])

    rendements_j = np.diff(pf) / pf[:-1]
    sharpe = (rendements_j.mean() / (rendements_j.std() + 1e-9)) * np.sqrt(252)

    return {
        'strategie_ret': round((pf[-1]/pf[0]-1)*100, 2),
        'bh_ret':        round((bh[-1]/bh[0]-1)*100, 2),
        'max_dd':        round(np.min(pf/np.maximum.accumulate(pf)-1)*100, 2),
        'sharpe':        round(sharpe, 3),
        'portefeuille':  pf,
        'bah':           bh,
    }


#  Prédictions
@torch.no_grad()
def predire(modele, loader):
    modele.eval()
    preds = []
    for X, _ in loader:
        preds.append(modele(X.to(device)).cpu().numpy())
    return np.vstack(preds)


#  Métriques
def calculer_metriques(p, t, nom_modele):
    """p, t : rendements log prédits / réels (vecteurs 1D)."""
    mae  = mean_absolute_error(t, p)
    rmse = np.sqrt(mean_squared_error(t, p))
    mape = np.mean(np.abs((t - p) / (np.abs(t) + 1e-8))) * 100
    dir_ = np.mean(np.sign(p) == np.sign(t)) * 100
    return {'Modele': nom_modele,
            'MAE':   round(mae, 6),
            'RMSE':  round(rmse, 6),
            'MAPE(%)': round(mape, 4),
            'DirAcc(%)': round(dir_, 2)}

print(' Fonctions utilitaires définies.')

 Fonctions utilitaires définies.


## 8. Classe d'Entraînement

In [21]:
class Entraineur:
    def __init__(self, modele, cfg, nom='modele'):
        self.modele   = modele.to(device)
        self.cfg      = cfg
        self.nom      = nom
        self.opt      = optim.Adam(modele.parameters(), lr=cfg['lr'], weight_decay=1e-5)
        self.critere  = nn.HuberLoss()
        self.meilleur_loss  = float('inf')
        self.meilleur_etat  = None

    def fit(self, loader_train, loader_val):
        if len(loader_val) == 0:
            return
        warmup = 5
        for epoch in range(self.cfg['epochs']):
            # Warmup du learning rate
            if epoch < warmup:
                for g in self.opt.param_groups:
                    g['lr'] = self.cfg['lr'] * (epoch+1) / warmup

            self.modele.train()
            for X, y in loader_train:
                X, y = X.to(device), y.to(device)
                self.opt.zero_grad()
                loss = self.critere(self.modele(X), y)
                loss.backward()
                nn.utils.clip_grad_norm_(self.modele.parameters(), 1.0)
                self.opt.step()

            self.modele.eval()
            v_loss = sum(
                self.critere(self.modele(X.to(device)), y.to(device)).item()
                for X, y in loader_val
            ) / len(loader_val)

            if v_loss < self.meilleur_loss:
                self.meilleur_loss = v_loss
                self.meilleur_etat = copy.deepcopy(self.modele.state_dict())

        if self.meilleur_etat:
            self.modele.load_state_dict(self.meilleur_etat)

print(' Classe Entraineur définie.')

 Classe Entraineur définie.


## 9. Pipeline Principal — Entraînement sur Tous les Actifs
> 3 modèles × N actifs. Sur CPU prévoir ~3–6 min/actif ☕

In [22]:
resultats_master = []
depot_tickers   = {}

print(' Entraînement avec isolation stricte des données...\n')

for ticker, (classe, nom) in tqdm(ALL_TICKERS.items()):
    try:
        df = ajouter_features(donnees_brutes[ticker], classe)
        n  = len(df)
        idx_tr = range(0, int(n*0.70))
        idx_vl = range(int(n*0.70), int(n*0.85))
        idx_te = range(int(n*0.85), n)

        data = preparer_donnees(df, CONFIG, idx_tr, idx_vl, idx_te)
        prix_test = df['Close'].iloc[idx_te].values  # prix réels alignés sur test

        depot_tickers[ticker] = {
            'df_feat':    df,
            'scaler':     data['scaler'],
            'cols_feats': data['cols_feats'],
            'prix_test':  prix_test,
            'ret_log_test': data['cibles_test'],
            'entraineurs': {},
            'preds_brutes': {},
        }

        architectures = {
            'LSTM':        ModeleLSTM(data['n_feats'],        output=CONFIG['horizon']).to(device),
            'BiLSTM':      ModeleBiLSTM(data['n_feats'],      output=CONFIG['horizon']).to(device),
            'Transformer': ModeleTransformer(data['n_feats'], output=CONFIG['horizon']).to(device),
        }

        for nom_modele, modele in architectures.items():
            entr = Entraineur(modele, CONFIG, nom=f'{ticker}-{nom_modele}')
            entr.fit(
                DataLoader(data['train'], batch_size=32, shuffle=True),
                DataLoader(data['val'],   batch_size=32)
            )

            mu = predire(modele, DataLoader(data['test'], batch_size=32))
            p_ret = mu[:, 0]
            t_ret = data['test'].y.numpy()[:, 0]

            metr = calculer_metriques(p_ret, t_ret, nom_modele)
            bt   = backtest(p_ret, prix_test)

            depot_tickers[ticker]['preds_brutes'][nom_modele] = {'p': p_ret, 't': t_ret}
            depot_tickers[ticker]['entraineurs'][nom_modele]  = entr

            entree = {'Ticker': ticker, 'Classe': classe, 'Nom': nom}
            entree.update(metr)
            entree.update({'StratRet(%)': bt['strategie_ret'],
                           'BH_Ret(%)':  bt['bh_ret'],
                           'MaxDD(%)':   bt['max_dd'],
                           'Sharpe':     bt['sharpe']})
            resultats_master.append(entree)

        # ── Ensemble : moyenne des prédictions ────────────────────────────
        ens_p = np.mean([depot_tickers[ticker]['preds_brutes'][m]['p']
                         for m in architectures], axis=0)
        ens_t = data['test'].y.numpy()[:, 0]

        metr_ens = calculer_metriques(ens_p, ens_t, 'Ensemble')
        bt_ens   = backtest(ens_p, prix_test)

        depot_tickers[ticker]['preds_brutes']['Ensemble'] = {'p': ens_p, 't': ens_t}

        entree_ens = {'Ticker': ticker, 'Classe': classe, 'Nom': nom}
        entree_ens.update(metr_ens)
        entree_ens.update({'StratRet(%)': bt_ens['strategie_ret'],
                           'BH_Ret(%)':  bt_ens['bh_ret'],
                           'MaxDD(%)':   bt_ens['max_dd'],
                           'Sharpe':     bt_ens['sharpe']})
        resultats_master.append(entree_ens)

    except Exception as e:
        print(f' Erreur sur {ticker}: {e}')

df_master = pd.DataFrame(resultats_master)
df_master.to_csv(CONFIG['dossier'] + 'resultats_master.csv', index=False)
print(f'\ Entraînement terminé — {len(depot_tickers)} actifs traités.')
display(df_master.sort_values('DirAcc(%)', ascending=False).head(10))

 Entraînement avec isolation stricte des données...



100%|██████████| 21/21 [12:28<00:00, 35.66s/it]

\ Entraînement terminé — 21 actifs traités.


,Ticker,Classe,Nom,Modele,MAE,RMSE,MAPE(%),DirAcc(%),StratRet(%),BH_Ret(%),MaxDD(%),Sharpe
43,GBPUSD=X,Forex,GBP/USD,Ensemble,0.003051,0.003984,121.043098,66.08,-10.34,4.67,-11.76,-2.872
42,GBPUSD=X,Forex,GBP/USD,Transformer,0.003065,0.003931,196.626907,63.88,-11.13,4.67,-11.41,-2.433
41,GBPUSD=X,Forex,GBP/USD,BiLSTM,0.003149,0.004094,131.557297,62.56,-2.92,4.67,-5.96,-0.582
50,AUDUSD=X,Forex,AUD/USD,Transformer,0.003892,0.005155,156.441193,60.79,-0.64,15.35,-3.90,-0.141
79,^GDAXI,Indices,DAX,Ensemble,0.007321,0.010037,107.319397,59.09,2.98,9.22,-6.06,0.469
78,^GDAXI,Indices,DAX,Transformer,0.007544,0.010247,144.258606,55.91,2.34,9.22,-3.82,0.441
39,EURUSD=X,Forex,EUR/USD,Ensemble,0.002970,0.003931,121.897400,55.07,-1.68,8.52,-4.88,-0.387
51,AUDUSD=X,Forex,AUD/USD,Ensemble,0.004037,0.005251,125.032898,55.07,1.92,15.35,-3.02,0.663
46,USDJPY=X,Forex,USD/JPY,Transformer,0.003943,0.005175,133.119507,55.07,-8.58,5.39,-8.58,-1.250
64,NG=F,Matières Prem.,Gaz naturel,LSTM,0.036145,0.067385,115.324097,55.05,-28.06,-20.82,-59.37,0.155


## 10. Analyse Comparative Multi-Actifs

In [23]:
df_ens = df_master[df_master['Modele'] == 'Ensemble'].copy()

print(' RÉSULTATS ENSEMBLE — RÉSUMÉ')
print('='*80)
print(df_ens[['Ticker','Classe','DirAcc(%)','MAPE(%)','Sharpe','StratRet(%)','BH_Ret(%)','MaxDD(%)']].to_string(index=False))

print('\n MOYENNE PAR CLASSE D\'ACTIF')
print('='*60)
class_avg = df_ens.groupby('Classe')[['DirAcc(%)','MAPE(%)','Sharpe','StratRet(%)']].mean().round(2)
class_avg = class_avg.sort_values('DirAcc(%)', ascending=False)
print(class_avg.to_string())

 RÉSULTATS ENSEMBLE — RÉSUMÉ
  Ticker         Classe  DirAcc(%)      MAPE(%)  Sharpe  StratRet(%)  BH_Ret(%)  MaxDD(%)
    TSLA        Actions      48.39   220.400406  -0.681       -16.18      90.02    -27.34
    NVDA        Actions      53.92   181.849197   2.084        91.92     141.49    -17.43
    MSFT        Actions      48.39   278.150696  -1.313        -8.78      15.29    -12.00
     JPM        Actions      51.61   121.602097   1.332        19.75      42.58     -5.39
     XOM        Actions      49.31 49300.937500   0.383         3.68      53.52    -15.68
 BTC-USD         Crypto      48.69   222.725998  -2.520       -26.69       3.63    -29.33
 ETH-USD         Crypto      50.73  1220.295166  -0.223       -13.38      44.66    -33.83
 SOL-USD         Crypto      51.65   163.438004  -0.474       -23.79     -39.70    -44.68
 BNB-USD         Crypto      49.85   241.077194  -0.422        -7.59      22.24    -17.81
EURUSD=X          Forex      55.07   121.897400  -0.387        -1.68   

In [24]:
#  Carte de chaleur : précision directionnelle & MAPE
pivot_dir  = df_master[df_master['Modele']=='Ensemble'].pivot_table(
                index='Classe', columns='Ticker', values='DirAcc(%)')
pivot_mape = df_master[df_master['Modele']=='Ensemble'].pivot_table(
                index='Classe', columns='Ticker', values='MAPE(%)')

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Précision directionnelle (%) — Plus haut = meilleur',
                    'MAPE (%) — Plus bas = meilleur'])

fig.add_trace(go.Heatmap(
    z=pivot_dir.values, x=pivot_dir.columns, y=pivot_dir.index,
    colorscale='RdYlGn', zmin=45, zmax=60,
    text=np.round(pivot_dir.values,1), texttemplate='%{text}', showscale=True
), row=1, col=1)

fig.add_trace(go.Heatmap(
    z=pivot_mape.values, x=pivot_mape.columns, y=pivot_mape.index,
    colorscale='RdYlGn_r', zmin=0, zmax=15,
    text=np.round(pivot_mape.values,2), texttemplate='%{text}', showscale=True
), row=1, col=2)

fig.update_layout(height=420, template='plotly_dark',
    title='Carte de chaleur — Prédictabilité (Modèle Ensemble)')
fig.show()

In [25]:
#  Comparaison des architectures
COULEURS_MODELES = {'LSTM':'#3498db','BiLSTM':'#2ecc71',
                    'Transformer':'#e74c3c','Ensemble':'#FFD700'}
metriques_arch = ['DirAcc(%)','MAPE(%)','Sharpe']

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Précision directionnelle (%)','MAPE (%)','Sharpe'])

for col_i, met in enumerate(metriques_arch, 1):
    moy_mod = df_master.groupby('Modele')[met].mean()
    asc = met == 'MAPE(%)'
    moy_mod = moy_mod.sort_values(ascending=asc)
    for mnom, val in moy_mod.items():
        fig.add_trace(go.Bar(
            x=[mnom], y=[val], name=mnom,
            marker_color=COULEURS_MODELES.get(mnom,'#aaa'),
            showlegend=(col_i==1),
            text=[f'{val:.2f}'], textposition='outside'
        ), row=1, col=col_i)

fig.update_layout(height=440, template='plotly_dark', barmode='group',
    title='Comparaison des Architectures — Moyenne tous actifs')
fig.show()

## 11. Backtesting : Stratégie ML vs Achat-Conservation

In [26]:
df_bt = df_ens[['Ticker','Classe','StratRet(%)','BH_Ret(%)','MaxDD(%)','Sharpe']].copy()
df_bt['Alpha(%)'] = df_bt['StratRet(%)'] - df_bt['BH_Ret(%)']
df_bt = df_bt.sort_values('StratRet(%)', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(x=df_bt['Ticker'], y=df_bt['StratRet(%)'],
    name='Stratégie ML', marker_color='#2ecc71',
    text=[f'{v:.1f}%' for v in df_bt['StratRet(%)']], textposition='outside'))
fig.add_trace(go.Bar(x=df_bt['Ticker'], y=df_bt['BH_Ret(%)'],
    name='Achat-Conservation', marker_color='#3498db', opacity=0.65,
    text=[f'{v:.1f}%' for v in df_bt['BH_Ret(%)']], textposition='outside'))
fig.update_layout(title='Rendement Stratégie vs Achat-Conservation — Modèle Ensemble',
    barmode='group', height=500, template='plotly_dark',
    xaxis_title='Actif', yaxis_title='Rendement (%)')
fig.show()

print('\n RÉCAPITULATIF BACKTEST')
print(df_bt.to_string(index=False))


 RÉCAPITULATIF BACKTEST
  Ticker         Classe  StratRet(%)  BH_Ret(%)  MaxDD(%)  Sharpe  Alpha(%)
    NVDA        Actions        91.92     141.49    -17.43   2.084    -49.57
    SI=F Matières Prem.        24.40     187.74     -1.15   1.737   -163.34
     JPM        Actions        19.75      42.58     -5.39   1.332    -22.83
    GC=F Matières Prem.        16.68      58.51     -6.76   1.184    -41.83
   ^GSPC        Indices        14.81      48.18     -4.93   1.242    -33.37
     XOM        Actions         3.68      53.52    -15.68   0.383    -49.84
  ^GDAXI        Indices         2.98       9.22     -6.06   0.469     -6.24
AUDUSD=X          Forex         1.92      15.35     -3.02   0.663    -13.43
   ^N225        Indices         0.62      85.47    -10.65   0.126    -84.85
EURUSD=X          Forex        -1.68       8.52     -4.88  -0.387    -10.20
    CL=F Matières Prem.        -5.27      66.67    -16.00  -0.305    -71.94
USDJPY=X          Forex        -6.43       5.39     -6.60  -1.0

In [27]:
#  Courbes de portefeuille — meilleur actif par classe
meilleur_par_classe = (
    df_ens.loc[df_ens.groupby('Classe')['DirAcc(%)'].idxmax()]['Ticker'].tolist()
)

fig = go.Figure()
for ticker in meilleur_par_classe:
    if ticker not in depot_tickers:
        continue
    rp    = depot_tickers[ticker]['preds_brutes']['Ensemble']
    prix  = depot_tickers[ticker]['prix_test']
    bt    = backtest(rp['p'], prix)
    cls   = ALL_TICKERS[ticker][0]
    coul  = COULEURS_CLASSES[cls]
    norm_pf = bt['portefeuille'] / bt['portefeuille'][0] * 100
    norm_bh = bt['bah']          / bt['bah'][0]          * 100
    fig.add_trace(go.Scatter(y=norm_pf, name=f'{ticker} Stratégie',
        line=dict(color=coul, width=2)))
    fig.add_trace(go.Scatter(y=norm_bh, name=f'{ticker} A-C',
        line=dict(color=coul, width=1, dash='dot'), opacity=0.5))

fig.update_layout(
    title='Meilleur actif par classe — Stratégie vs Achat-Conservation (base=100)',
    height=500, template='plotly_dark',
    xaxis_title='Jours (période de test)', yaxis_title='Valeur du portefeuille (base=100)')
fig.show()

## 12. Prévisions Futures — Tous les Actifs

In [28]:
@torch.no_grad()
def prevoir_futur(modele, df_feat, cols, scaler, seq_len):
    """Retourne les rendements log prédits pour J+1 à J+horizon."""
    modele.eval()
    seq = scaler.transform(df_feat[cols].values[-seq_len:])
    X   = torch.FloatTensor(seq).unsqueeze(0).to(device)
    ps  = modele(X).cpu().numpy().reshape(-1)
    return ps.tolist()

print(' PRÉVISIONS ENSEMBLE — Prochains jours\n')
print(f'{"Actif":<12} {"Classe":<15} {"Dernier prix":>14}  Rendement J+1 prédit')
print('-'*65)

toutes_previsions = {}
for ticker, depot in depot_tickers.items():
    classe, _  = ALL_TICKERS[ticker]
    df_feat    = depot['df_feat']
    cols       = depot['cols_feats']
    scaler     = depot['scaler']

    preds_par_modele = [
        prevoir_futur(entr.modele, df_feat, cols, scaler, CONFIG['seq_len'])
        for entr in depot['entraineurs'].values()
    ]
    prev_log = np.mean(preds_par_modele, axis=0).tolist()
    toutes_previsions[ticker] = prev_log

    dernier_p = df_feat['Close'].iloc[-1]
    chg       = (np.exp(prev_log[0]) - 1) * 100
    print(f'{ticker:<12} {classe:<15} {dernier_p:>14.4f}  {chg:+.2f}%')

print()

 PRÉVISIONS ENSEMBLE — Prochains jours

Actif        Classe            Dernier prix  Rendement J+1 prédit
-----------------------------------------------------------------
TSLA         Actions               443.3000  -0.16%
NVDA         Actions               235.7400  +0.81%
MSFT         Actions               409.4300  -0.31%
JPM          Actions               299.9100  +0.10%
XOM          Actions               151.7500  +0.29%
BTC-USD      Crypto              81051.2500  +0.04%
ETH-USD      Crypto               2280.9348  +0.07%
SOL-USD      Crypto                 92.1487  +0.26%
BNB-USD      Crypto                677.9398  +0.29%
EURUSD=X     Forex                   1.1716  -0.06%
GBPUSD=X     Forex                   1.3526  +0.07%
USDJPY=X     Forex                 157.8510  -0.07%
AUDUSD=X     Forex                   0.7255  -0.11%
GC=F         Matières Prem.       4678.1001  -0.04%
CL=F         Matières Prem.        101.1700  -0.85%
SI=F         Matières Prem.         84.9120  -0.

In [29]:
#  Signaux haussiers / baissiers
haussiers, baissiers = [], []
for ticker, prev in toutes_previsions.items():
    chg = (np.exp(prev[0]) - 1) * 100
    cls = ALL_TICKERS[ticker][0]
    entree = {'ticker': ticker, 'classe': cls, 'chg_j1': round(chg, 2)}
    (haussiers if chg > 0 else baissiers).append(entree)

print(' SIGNAUX HAUSSIERS (J+1) :')
for e in sorted(haussiers, key=lambda x: -x['chg_j1']):
    print(f"   ▲ {e['ticker']:<12} [{e['classe']:<15}]  +{e['chg_j1']:.2f}%")

print('\nSIGNAUX BAISSIERS (J+1) :')
for e in sorted(baissiers, key=lambda x: x['chg_j1']):
    print(f"   ▼ {e['ticker']:<12} [{e['classe']:<15}]  {e['chg_j1']:.2f}%")

print('\n Ces prévisions sont issues de modèles statistiques. Elles ne constituent PAS des conseils financiers.')

 SIGNAUX HAUSSIERS (J+1) :
   ▲ NVDA         [Actions        ]  +0.81%
   ▲ XOM          [Actions        ]  +0.29%
   ▲ BNB-USD      [Crypto         ]  +0.29%
   ▲ SOL-USD      [Crypto         ]  +0.26%
   ▲ NG=F         [Matières Prem. ]  +0.14%
   ▲ JPM          [Actions        ]  +0.10%
   ▲ ETH-USD      [Crypto         ]  +0.07%
   ▲ GBPUSD=X     [Forex          ]  +0.07%
   ▲ BTC-USD      [Crypto         ]  +0.04%

SIGNAUX BAISSIERS (J+1) :
   ▼ SI=F         [Matières Prem. ]  -0.87%
   ▼ CL=F         [Matières Prem. ]  -0.85%
   ▼ ^IXIC        [Indices        ]  -0.39%
   ▼ MSFT         [Actions        ]  -0.31%
   ▼ ^GDAXI       [Indices        ]  -0.22%
   ▼ TSLA         [Actions        ]  -0.16%
   ▼ AUDUSD=X     [Forex          ]  -0.11%
   ▼ USDJPY=X     [Forex          ]  -0.07%
   ▼ EURUSD=X     [Forex          ]  -0.06%
   ▼ ^N225        [Indices        ]  -0.06%
   ▼ ^GSPC        [Indices        ]  -0.05%
   ▼ GC=F         [Matières Prem. ]  -0.04%

 Ces prévisions sont 

## 13. Classement Final & Insights

In [30]:
score_df = df_ens[['Ticker','Classe','Nom','DirAcc(%)','MAPE(%)','Sharpe','StratRet(%)','BH_Ret(%)']].copy()

score_df['rang_dir']    = score_df['DirAcc(%)'].rank(ascending=False)
score_df['rang_mape']   = score_df['MAPE(%)'].rank(ascending=True)
score_df['rang_sharpe'] = score_df['Sharpe'].rank(ascending=False)
score_df['score_composé'] = (score_df['rang_dir'] + score_df['rang_mape'] + score_df['rang_sharpe']) / 3

score_df = score_df.sort_values('score_composé')
score_df['Rang'] = range(1, len(score_df)+1)

print(' CLASSEMENT DE PRÉDICTABILITÉ (score bas = plus prévisible)')
print('='*75)
cols_aff = ['Rang','Ticker','Classe','DirAcc(%)','MAPE(%)','Sharpe','score_composé']
print(score_df[cols_aff].to_string(index=False))

score_df.to_csv(CONFIG['dossier'] + 'classement_final.csv', index=False)
print(f'\n Sauvegardé : {CONFIG["dossier"]}classement_final.csv')

 CLASSEMENT DE PRÉDICTABILITÉ (score bas = plus prévisible)
 Rang   Ticker         Classe  DirAcc(%)      MAPE(%)  Sharpe  score_composé
    1   ^GDAXI        Indices      59.09   107.319397   0.469       3.666667
    2      JPM        Actions      51.61   121.602097   1.332       5.333333
    3 AUDUSD=X          Forex      55.07   125.032898   0.663       5.500000
    4    ^GSPC        Indices      53.92   147.681000   1.242       6.166667
    5     NG=F Matières Prem.      50.92   106.122498   0.365       6.333333
    6     NVDA        Actions      53.92   181.849197   2.084       6.500000
    7 EURUSD=X          Forex      55.07   121.897400  -0.387       7.500000
    8 GBPUSD=X          Forex      66.08   121.043098  -2.872       8.666667
    9     GC=F Matières Prem.      48.17   140.119003   1.184       9.666667
   10     SI=F Matières Prem.      44.50   151.439499   1.737      10.666667
   11  SOL-USD         Crypto      51.65   163.438004  -0.474      11.000000
   12 USDJPY=X  

In [31]:
# ── Graphique de classement ───────────────────────────────────────────────
fig = px.bar(
    score_df.sort_values('score_composé'),
    x='Ticker', y='DirAcc(%)',
    color='Classe',
    text='DirAcc(%)',
    color_discrete_map=COULEURS_CLASSES,
    template='plotly_dark',
    title=' Classement de Précision Directionnelle — Modèle Ensemble',
)
fig.add_hline(y=50, line_dash='dash', line_color='white', opacity=0.5,
              annotation_text='Référence aléatoire 50%')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, xaxis_tickangle=-45)
fig.write_html(CONFIG['dossier'] + 'classement.html')
fig.show()

## 14. Tableau de Bord des Signaux à Haute Conviction

In [32]:
signaux = []
for ticker, prev in toutes_previsions.items():
    stats_ticker = df_ens[df_ens['Ticker'] == ticker]
    if stats_ticker.empty:
        continue
    stats_ticker = stats_ticker.iloc[0]
    chg = (np.exp(prev[0]) - 1) * 100

    if stats_ticker['DirAcc(%)'] > 51.5:
        signaux.append({
            'Ticker':      ticker,
            'Classe':      stats_ticker['Classe'],
            'Précision':   stats_ticker['DirAcc(%)'],
            'Mouvement J+1': chg,
            'Confiance':   'HAUTE' if stats_ticker['DirAcc(%)'] > 53 else 'MOY',
        })

df_signaux = pd.DataFrame(signaux).sort_values('Précision', ascending=False)

if not df_signaux.empty:
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=['<b>Ticker</b>','<b>Classe</b>','<b>Précision Modèle</b>',
                    '<b>Prévision J+1</b>','<b>Confiance</b>'],
            fill_color='#2c3e50', font=dict(color='white'), align='left'
        ),
        cells=dict(
            values=[df_signaux.Ticker, df_signaux.Classe,
                    df_signaux['Précision'].map('{:.2f}%'.format),
                    df_signaux['Mouvement J+1'].map('{:+.2f}%'.format),
                    df_signaux.Confiance],
            fill_color='#34495e', font=dict(color='white'), align='left'
        )
    )])
    fig.update_layout(title=' Signaux de Trading (Prédictabilité × Momentum)',
                      template='plotly_dark')
    fig.show()
else:
    print('Aucun signal à haute conviction détecté.')

## 15. Rapport Final

In [33]:
print('='*70)
print('    PRÉVISION MULTI-ACTIFS PAR DEEP LEARNING — RAPPORT FINAL')
print('='*70)
print(f'  Actifs entraînés : {len(depot_tickers)}')
print(f'  Modèles/actif   : LSTM + BiLSTM + Transformer + Ensemble')
print(f'  Période         : {CONFIG["date_debut"]} → {CONFIG["date_fin"]}')
print(f'  Horizon         : {CONFIG["horizon"]} jour(s)')
print()

print('   ACTIFS LES PLUS PRÉVISIBLES (par DirAcc) :')
for _, row in score_df.head(3).iterrows():
    print(f'     #{int(row["Rang"])} {row["Ticker"]:<10} [{row["Classe"]}]  '
          f'DirAcc: {row["DirAcc(%)"]:.1f}%  MAPE: {row["MAPE(%)"]:.2f}%')

print()
print('  MEILLEURS ALPHA (Stratégie − Achat-Conservation) :')
df_alpha = df_bt.sort_values('Alpha(%)', ascending=False).head(3)
for _, row in df_alpha.iterrows():
    print(f'     {row["Ticker"]:<10} Stratégie: {row["StratRet(%)"]:.1f}%  '
          f'A-C: {row["BH_Ret(%)"]:.1f}%  Alpha: {row["Alpha(%)"]:.1f}%')

print()
print('  PRÉDICTABILITÉ MOYENNE PAR CLASSE :')
for cls, row in class_avg.iterrows():
    barre = '█' * int(row['DirAcc(%)'] / 2)
    print(f'     {cls:<16} {barre:<30} {row["DirAcc(%)"]:.1f}%')

print()
print('  FICHIERS GÉNÉRÉS :')
print(f'     {CONFIG["dossier"]}resultats_master.csv  — toutes les métriques')
print(f'     {CONFIG["dossier"]}classement_final.csv  — classement de prédictabilité')
print(f'     {CONFIG["dossier"]}classement.html       — graphique interactif')
print('='*70)
print('  Uniquement à des fins de recherche. NON un conseil financier.')
print('='*70)

    PRÉVISION MULTI-ACTIFS PAR DEEP LEARNING — RAPPORT FINAL
  Actifs entraînés : 21
  Modèles/actif   : LSTM + BiLSTM + Transformer + Ensemble
  Période         : 2019-01-01 → 2026-05-15
  Horizon         : 1 jour(s)

   ACTIFS LES PLUS PRÉVISIBLES (par DirAcc) :
     #1 ^GDAXI     [Indices]  DirAcc: 59.1%  MAPE: 107.32%
     #2 JPM        [Actions]  DirAcc: 51.6%  MAPE: 121.60%
     #3 AUDUSD=X   [Forex]  DirAcc: 55.1%  MAPE: 125.03%

  MEILLEURS ALPHA (Stratégie − Achat-Conservation) :
     SOL-USD    Stratégie: -23.8%  A-C: -39.7%  Alpha: 15.9%
     NG=F       Stratégie: -10.6%  A-C: -20.8%  Alpha: 10.2%
     ^GDAXI     Stratégie: 3.0%  A-C: 9.2%  Alpha: -6.2%

  PRÉDICTABILITÉ MOYENNE PAR CLASSE :
     Forex            ███████████████████████████    55.7%
     Indices          █████████████████████████      50.4%
     Actions          █████████████████████████      50.3%
     Crypto           █████████████████████████      50.2%
     Matières Prem.   ███████████████████████       

---
## Références
| Ressource | Lien |
|---|---|
| LSTM pour séries temporelles | [github.com/jaungiers/LSTM](https://github.com/jaungiers/LSTM-Neural-Network-for-Time-Series-Prediction) |
| PyTorch Forecasting | [github.com/jdb78/pytorch-forecasting](https://github.com/jdb78/pytorch-forecasting) |
| Bibliothèque TA | [github.com/bukosabino/ta](https://github.com/bukosabino/ta) |
| yfinance | [github.com/ranaroussi/yfinance](https://github.com/ranaroussi/yfinance) |
| Mécanisme d'attention | [arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762) |

---
*Développé avec PyTorch · yfinance · ta · Plotly · scikit-learn*

⚠️ **Avertissement** : À des fins éducatives/de recherche uniquement. Les performances passées ne préjugent pas des résultats futurs.